In [39]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.head()

,ID,City A,City B,Distance,Time of Day,Weather,Traffic,Road Quality,Driver Experience,deliver_time
0,1,Satu Mare,Suceava,352,452,Fog,154.014691,370,30,355
1,2,Ploiesti,Timisoara,519,1386,Clear,949.697532,701,2,529
2,3,Deva,Bacau,457,91,Fog,387.019309,45,26,465
3,4,Hunedoara,Focsani,447,1120,Clear,130.544017,643,6,441
4,5,Hunedoara,Arad,201,1096,Clear,619.557737,375,20,230


In [40]:
#subtasks

subtask = len(df_test[(df_test["City A"] == 'Barlad') & (df_test["Weather"] == 'Fog')])

df_subtask = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": 1,
    "answer": [subtask]
})

df_subtask

,subtaskID,datapointID,answer
0,1,1,15


In [41]:
df_train.Weather.unique()

array(['Fog', 'Clear', 'Rain', 'Snow'], dtype=object)

In [42]:
#preprocessing

df_train.Weather = df_train.Weather.map({"Clear": 0, "Rain": 1, "Snow": 2, "Fog": 3})
df_test.Weather = df_test.Weather.map({"Clear": 0, "Rain": 1, "Snow": 2, "Fog": 3})

from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
df_train['City A'] = enc.fit_transform(df_train['City A'])
df_train['City B'] = enc.transform(df_train['City B'])
df_test['City A'] = enc.transform(df_test['City A'])
df_test['City B'] = enc.transform(df_test['City B'])

df_train.head()

,ID,City A,City B,Distance,Time of Day,Weather,Traffic,Road Quality,Driver Experience,deliver_time
0,1,25,28,352,452,3,154.014691,370,30,355
1,2,21,31,519,1386,0,949.697532,701,2,529
2,3,13,2,457,91,3,387.019309,45,26,465
3,4,17,15,447,1120,0,130.544017,643,6,441
4,5,17,1,201,1096,0,619.557737,375,20,230


In [43]:
df_train.columns

Index(['ID', 'City A', 'City B', 'Distance', 'Time of Day', 'Weather',
       'Traffic', 'Road Quality', 'Driver Experience', 'deliver_time'],
      dtype='object')

In [44]:
features = ['City A', 'City B', 'Distance', 'Time of Day', 'Weather', 'Traffic', 'Road Quality', 'Driver Experience']
target = 'deliver_time'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_final = pd.DataFrame(scaler.transform(X_final), columns=features)

X.head()

,City A,City B,Distance,Time of Day,Weather,Traffic,Road Quality,Driver Experience
0,0.866033,1.169855,-0.129339,-0.641373,1.340729,-1.179125,-0.463054,0.821636
1,0.459382,1.476488,0.773098,1.597257,-1.342339,1.572952,0.688027,-1.600130
2,-0.353918,-1.487629,0.438062,-1.506624,1.340729,-0.373218,-1.593269,0.475670
3,0.052732,-0.158887,0.384024,0.959703,-1.342339,-1.260304,0.486327,-1.254164
4,0.052732,-1.589839,-0.945315,0.902179,-1.342339,0.431077,-0.445666,-0.043280


In [45]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

model = LinearRegression()

cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error').mean()

-1.638061544738651

In [46]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [47]:
df_preds = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.ID,
    "answer": y_preds
})

df_preds.head()

,subtaskID,datapointID,answer
0,2,10001,386.810661
1,2,10002,243.041461
2,2,10003,480.925736
3,2,10004,248.444978
4,2,10005,336.742770


In [48]:
submission = pd.concat([df_subtask, df_preds])

submission

,subtaskID,datapointID,answer
0,1,1,15.000000
0,2,10001,386.810661
1,2,10002,243.041461
2,2,10003,480.925736
3,2,10004,248.444978
...,...,...,...
1995,2,11996,522.710626
1996,2,11997,430.140524
1997,2,11998,135.150540
1998,2,11999,373.387690


In [49]:
submission.to_csv("submission.csv", index=False)